# 10.3 Apply: 通用的“拆分-应用-合并”模式

In [18]:
import pandas as pd
import numpy as np
tips = pd.read_csv('../examples/tips.csv')
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,No,Sat,Dinner,2,0.098204


In [19]:
def top(df, n=5, column='tip_pct'):
    return df.sort_values(by=column, ascending=False).head(n)
top(tips,6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [20]:
tips.groupby('smoker').apply(top)

total_bill   tip   day    time  size   tip_pct
smoker                                                    
No     232       11.61  3.39   Sat  Dinner     2  0.291990
       149        7.51  2.00  Thur   Lunch     2  0.266312
       51        10.29  2.60   Sun  Dinner     2  0.252672
       185       20.69  5.00   Sun  Dinner     5  0.241663
       88        24.71  5.85  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15   Sun  Dinner     2  0.710345
       178        9.60  4.00   Sun  Dinner     2  0.416667
       67         3.07  1.00   Sat  Dinner     1  0.325733
       183       23.17  6.50   Sun  Dinner     4  0.280535
       109       14.31  4.00   Sat  Dinner     2  0.279525

In [21]:
tips.groupby(['smoker', 'day']).apply(top, n=1, column='total_bill')

total_bill    tip    time  size   tip_pct
smoker day                                                
No     Fri  94        22.75   3.25  Dinner     2  0.142857
       Sat  212       48.33   9.00  Dinner     4  0.186220
       Sun  156       48.17   5.00  Dinner     6  0.103799
       Thur 142       41.19   5.00   Lunch     5  0.121389
Yes    Fri  95        40.17   4.73  Dinner     4  0.117750
       Sat  170       50.81  10.00  Dinner     3  0.196812
       Sun  182       45.35   3.50  Dinner     3  0.077178
       Thur 197       43.11   5.00   Lunch     4  0.115982

In [22]:
# 传入group_keys=False可以不在结果中包含分组键

## 10.3.1 分位数和桶分析

In [23]:
frame = pd.DataFrame({'data1': np.random.standard_normal(1000).round(3),
                      'data2': np.random.standard_normal(1000).round(3)})
frame.head()

,data1,data2
0,-0.420,-0.972
1,-1.885,0.594
2,-0.602,0.485
3,1.343,-1.784
4,-0.315,-1.451


In [24]:
quartiles = pd.cut(frame.data1, 4)  # 区间长度等分
quartiles[:10]

0     (-1.381, 0.135]
1    (-2.903, -1.381]
2     (-1.381, 0.135]
3      (0.135, 1.651]
4     (-1.381, 0.135]
5      (1.651, 3.167]
6     (-1.381, 0.135]
7    (-2.903, -1.381]
8     (-1.381, 0.135]
9      (0.135, 1.651]
Name: data1, dtype: category
Categories (4, interval[float64, right]): [(-2.903, -1.381] < (-1.381, 0.135] < (0.135, 1.651] < (1.651, 3.167]]

In [26]:
def get_stats(group):
    return pd.DataFrame({'min': group.min(), 'max': group.max(),
            'count': group.count(), 'mean': group.mean()})
grouped = frame.groupby(quartiles)
grouped.apply(get_stats)

min    max  count      mean
data1                                                
(-2.903, -1.381] data1 -2.897 -1.383     96 -1.828167
                 data2 -3.345  2.258     96 -0.195781
(-1.381, 0.135]  data1 -1.379  0.135    462 -0.493823
                 data2 -2.729  3.018    462 -0.010294
(0.135, 1.651]   data1  0.140  1.628    400  0.715658
                 data2 -2.746  2.642    400 -0.002558
(1.651, 3.167]   data1  1.685  3.167     42  2.114881
                 data2 -1.763  2.004     42  0.077310

In [27]:
grouped.agg(['min', 'max', 'count', 'mean'])

data1                         data2                       
                    min    max count      mean    min    max count      mean
data1                                                                       
(-2.903, -1.381] -2.897 -1.383    96 -1.828167 -3.345  2.258    96 -0.195781
(-1.381, 0.135]  -1.379  0.135   462 -0.493823 -2.729  3.018   462 -0.010294
(0.135, 1.651]    0.140  1.628   400  0.715657 -2.746  2.642   400 -0.002557
(1.651, 3.167]    1.685  3.167    42  2.114881 -1.763  2.004    42  0.077310

In [28]:
quantiles_samp = pd.qcut(frame['data1'],4, labels=False)  # 区间内个数相同 labels=False返回整数索引而非区间
quantiles_samp.head()

0    1
1    0
2    1
3    3
4    1
Name: data1, dtype: int64

In [29]:
grouped = frame.groupby(quantiles_samp)
grouped.apply(get_stats)

min    max  count      mean
data1                                     
0     data1 -2.897 -0.702    250 -1.312100
      data2 -3.345  2.440    250 -0.102808
1     data1 -0.695 -0.006    251 -0.317410
      data2 -2.729  3.018    251  0.020845
2     data1 -0.002  0.629    250  0.292328
      data2 -2.382  2.482    250 -0.089008
3     data1  0.634  3.167    249  1.229120
      data2 -2.746  2.642    249  0.085924

## 10.3.2 示例：用指定分组的值填充缺失值

In [30]:
s = pd.Series(np.random.randn(6))
s[::2] = np.nan  # 切片 start stop step
s

0         NaN
1   -1.670509
2         NaN
3    0.320218
4         NaN
5   -0.116517
dtype: float64

In [31]:
s.fillna(s.mean())

0   -0.488936
1   -1.670509
2   -0.488936
3    0.320218
4   -0.488936
5   -0.116517
dtype: float64

In [32]:
states = ["Ohio", "New York", "Vermont", "Florida",
          "Oregon", "Nevada", "California", "Idaho"]
group_key = ["East", "East", "East", "East",
             "West", "West", "West", "West"]
data = pd.Series(np.random.standard_normal(8), index=states)
data

Ohio          0.942424
New York     -0.397349
Vermont      -0.548970
Florida       0.895526
Oregon        1.229761
Nevada       -0.914993
California   -1.234220
Idaho        -0.001289
dtype: float64

In [33]:
fill_values = {'East':0.5, 'West':-1}
def fill_mean(group):
    key = group.name  # group.name属性包含分组的名字
    return group.fillna(fill_values[key])
data.groupby(group_key).apply(fill_mean)

East  Ohio          0.942424
      New York     -0.397349
      Vermont      -0.548970
      Florida       0.895526
West  Oregon        1.229761
      Nevada       -0.914993
      California   -1.234220
      Idaho        -0.001289
dtype: float64

## 10.3.3 示例：分组加权平均和随机采样

In [34]:
df = pd.DataFrame({"category": ["a", "a", "a", "a",
                                "b", "b", "b", "b"],
                   "data": np.random.standard_normal(8),
                   "weights": np.random.uniform(size=8)})
df

,category,data,weights
0,a,-1.509724,0.968048
1,a,1.358420,0.399463
2,a,-0.256057,0.354984
3,a,0.884029,0.070774
4,b,-0.538235,0.855814
5,b,0.457945,0.486757
6,b,0.326669,0.521570
7,b,-0.909353,0.974130


In [35]:
grouped = df.groupby("category")
def get_wavg(group):
    return np.average(group.data, weights=group.weights)
grouped.apply(get_wavg)

category
a   -0.528185
b   -0.335827
dtype: float64

In [36]:
close_px = pd.read_csv('../examples/stock_px.csv', parse_dates=True, index_col=0)
close_px.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2214 entries, 2003-01-02 to 2011-10-14
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2214 non-null   float64
 1   MSFT    2214 non-null   float64
 2   XOM     2214 non-null   float64
 3   SPX     2214 non-null   float64
dtypes: float64(4)
memory usage: 86.5 KB


In [41]:
def get_year(x):
    return x.year
rets = close_px.pct_change().dropna()
rets

,AAPL,MSFT,XOM,SPX
2003-01-03,0.006757,0.001421,0.000684,-0.000484
2003-01-06,0.000000,0.017975,0.024624,0.022474
2003-01-07,-0.002685,0.019052,-0.033712,-0.006545
2003-01-08,-0.020188,-0.028272,-0.004145,-0.014086
2003-01-09,0.008242,0.029094,0.021159,0.019386
...,...,...,...,...
2011-10-10,0.051406,0.026286,0.036977,0.034125
2011-10-11,0.029526,0.002227,-0.000131,0.000544
2011-10-12,0.004747,-0.001481,0.011669,0.009795
2011-10-13,0.015515,0.008160,-0.010238,-0.002974


In [48]:
by_year = rets.groupby(get_year)
for name,group in by_year:
    print(name)
    print(group)

2003
                AAPL      MSFT       XOM       SPX
2003-01-03  0.006757  0.001421  0.000684 -0.000484
2003-01-06  0.000000  0.017975  0.024624  0.022474
2003-01-07 -0.002685  0.019052 -0.033712 -0.006545
2003-01-08 -0.020188 -0.028272 -0.004145 -0.014086
2003-01-09  0.008242  0.029094  0.021159  0.019386
...              ...       ...       ...       ...
2003-12-24  0.030303 -0.003717  0.002080 -0.001807
2003-12-26  0.018627  0.006063  0.005336  0.001691
2003-12-29  0.017324  0.009272  0.013270  0.012401
2003-12-30  0.006623  0.002297  0.002619  0.000144
2003-12-31  0.004699 -0.005500  0.007837  0.002055

[251 rows x 4 columns]
2004
                AAPL      MSFT       XOM       SPX
2004-01-02 -0.004677  0.002765 -0.008929 -0.003094
2004-01-05  0.042293  0.025276  0.023249  0.012395
2004-01-06 -0.003607  0.003586 -0.006816  0.001292
2004-01-07  0.022624 -0.001340 -0.007149  0.002367
2004-01-08  0.033628 -0.001342 -0.002592  0.004963
...              ...       ...       ...       .

In [47]:
def spx_corr(group):
    return group.corrwith(group['SPX'])
by_year.apply(spx_corr)

,AAPL,MSFT,XOM,SPX
2003,0.541124,0.745174,0.661265,1.0
2004,0.374283,0.588531,0.557742,1.0
2005,0.467540,0.562374,0.631010,1.0
2006,0.428267,0.406126,0.518514,1.0
2007,0.508118,0.658770,0.786264,1.0
2008,0.681434,0.804626,0.828303,1.0
2009,0.707103,0.654902,0.797921,1.0
2010,0.710105,0.730118,0.839057,1.0
2011,0.691931,0.800996,0.859975,1.0


## 10.3.4 示例：分组线性回归

In [49]:
import statsmodels.api as sm
def regress(data, yvar, xvars):
    Y = data[yvar]
    X = data[xvars]
    X['intercept'] = 1.
    result = sm.OLS(Y, X).fit()  # 普通最小二乘回归
    return result.params
by_year.apply(regress, yvar='AAPL', xvars=['SPX'])

,SPX,intercept
2003,1.195406,0.000710
2004,1.363463,0.004201
2005,1.766415,0.003246
2006,1.645496,0.000080
2007,1.198761,0.003438
2008,0.968016,-0.001110
2009,0.879103,0.002954
2010,1.052608,0.001261
2011,0.806605,0.001514


# End